<a href="https://colab.research.google.com/github/Shubham4521/Cost-Aware-LLM-Router/blob/main/notebooks/02_run_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q openai tavily-python pandas

In [3]:
from google.colab import drive
import os
drive.mount('/content/drive')
OUT = "/content/drive/MyDrive/llm_router"
os.makedirs(OUT, exist_ok=True)
print("Saving to", OUT)

Mounted at /content/drive
Saving to /content/drive/MyDrive/llm_router


In [5]:
GITHUB_USER = "Shubham4521"
FREE_ID     = "openai/gpt-oss-120b"        # your Groq model from Phase 0
CHEAP_ID    = "openai/gpt-5-mini"
PREMIUM_ID  = "anthropic/claude-sonnet-5"
TEST_RUN    = True    # True = run 6 prompts only. Change to False later.

In [6]:
import time, json, re, subprocess, tempfile, random
import pandas as pd
from google.colab import userdata
from openai import OpenAI
from tavily import TavilyClient

df = pd.read_csv(
    f"https://raw.githubusercontent.com/{GITHUB_USER}/cost-aware-llm-router/main/data/eval_set.csv",
    keep_default_na=False)
df["needs_search"] = df["needs_search"].astype(str) == "True"

groq = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=userdata.get("GROQ_API_KEY"))
openrouter = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=userdata.get("OPENROUTER_API_KEY"))
tavily = TavilyClient(api_key=userdata.get("TAVILY_API_KEY"))

MODELS = {
    "free":    (groq, FREE_ID),
    "cheap":   (openrouter, CHEAP_ID),
    "premium": (openrouter, PREMIUM_ID),
    "judge":   (openrouter, PREMIUM_ID),   # premium model grades answers
}
TIERS = ["free", "cheap", "premium"]

def chat(name, messages, max_tokens=2000):
    client, model = MODELS[name]
    kwargs = dict(model=model, messages=messages, max_tokens=max_tokens)
    if client is openrouter:
        kwargs["extra_body"] = {"usage": {"include": True}}  # asks OpenRouter to report cost
    for attempt in range(6):
        try:
            start = time.time()
            r = client.chat.completions.create(**kwargs)
            return {
                "answer": r.choices[0].message.content or "",
                "input_tokens": r.usage.prompt_tokens,
                "output_tokens": r.usage.completion_tokens,
                "cost_usd": float(getattr(r.usage, "cost", 0) or 0),  # Groq free = 0
                "latency_s": round(time.time() - start, 2),
            }
        except Exception as e:
            if "429" in str(e) and attempt < 5:
                wait = 20 * (attempt + 1)
                print(f"Rate limited on {name}, waiting {wait}s")
                time.sleep(wait)
            else:
                raise

CACHE = f"{OUT}/search_cache.json"
search_cache = json.load(open(CACHE)) if os.path.exists(CACHE) else {}

def search_context(q):
    if q not in search_cache:
        res = tavily.search(q, max_results=5)
        search_cache[q] = "\n\n".join(f"{x['title']}\n{x['content']}" for x in res["results"])
        json.dump(search_cache, open(CACHE, "w"))
    return search_cache[q]

def build_messages(row):
    if row["needs_search"]:
        content = (f"Use these search results to answer.\n\nSEARCH RESULTS:\n"
                   f"{search_context(row['prompt'])}\n\nQUESTION: {row['prompt']}")
    else:
        content = row["prompt"]
    return [{"role": "user", "content": content}]

print("Loaded", len(df), "prompts")

Loaded 165 prompts


In [7]:
RAW = f"{OUT}/raw_results.csv"
run_df = df.groupby("category").head(1) if TEST_RUN else df

done = set()
if os.path.exists(RAW):
    prev = pd.read_csv(RAW, keep_default_na=False)
    done = set(zip(prev["id"], prev["tier"]))

total = len(run_df) * len(TIERS)
count = 0
for _, row in run_df.iterrows():
    for tier in TIERS:
        if (row["id"], tier) in done:
            continue
        try:
            res = chat(tier, build_messages(row))
        except Exception as e:
            print("FAILED", row["id"], tier, str(e)[:150])
            continue
        rec = {"id": row["id"], "tier": tier, "model": MODELS[tier][1], **res}
        pd.DataFrame([rec]).to_csv(RAW, mode="a", header=not os.path.exists(RAW), index=False)
        count += 1
        if count % 10 == 0:
            print(f"{count} new calls done")

print("Finished this run. New calls:", count, "| Planned:", total)

10 new calls done
Finished this run. New calls: 18 | Planned: 18


In [8]:
def extract_math(ans):
    m = re.findall(r"ANSWER:\s*(.+)", ans)
    if m:
        return m[-1].strip()
    lines = ans.strip().splitlines()
    return lines[-1] if lines else ""

def norm_math(s):
    s = str(s)
    s = re.sub(r"\\boxed\{(.*)\}", r"\1", s)
    s = re.sub(r"\\text\{(.*?)\}", r"\1", s)
    for t in ["$", "\\left", "\\right", "\\!", " "]:
        s = s.replace(t, "")
    s = s.replace("dfrac", "frac").replace("tfrac", "frac").rstrip(".")
    return s.lower()

def extract_letter(ans):
    a = ans.strip()
    m = re.match(r"^\W*([A-E1-5])\b", a)
    if m: return m.group(1)
    m = re.findall(r"answer\s*(?:is|:)?\s*\(?([A-E1-5])\b", a, re.I)
    if m: return m[-1].upper()
    m = re.findall(r"\b([A-E])\b", a)
    return m[-1] if m else ""

def run_code(ans, extra_json):
    ex = json.loads(extra_json)
    blocks = re.findall(r"```(?:python)?\n(.*?)```", ans, re.S)
    code = blocks[0] if blocks else ans
    prog = ("from typing import *\nimport math, re, collections, itertools, heapq, string\n"
            + code + "\n\n" + ex["test"] + f"\n\ncheck({ex['entry_point']})\n")
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
        f.write(prog)
        path = f.name
    try:
        return subprocess.run(["python", path], capture_output=True, timeout=15).returncode == 0
    except subprocess.TimeoutExpired:
        return False

JUDGE_PROMPT = """You are grading an AI answer. Score it from 1 to 10 for correctness, completeness and how well it follows the instructions.
{context}
TASK:
{task}

ANSWER:
{answer}

Reply with only a number from 1 to 10."""

def judge(task, answer, context=""):
    out = chat("judge", [{"role": "user", "content":
               JUDGE_PROMPT.format(task=task, answer=answer, context=context)}], max_tokens=500)
    nums = re.findall(r"\b(10|[1-9])\b", out["answer"])
    return (int(nums[0]) if nums else 0), out["cost_usd"]

def score(row, ans):
    t = row["answer_type"]
    if t == "math_answer":
        pred = extract_math(ans)
        if norm_math(pred) == norm_math(row["reference"]):
            return 10, 0.0
        q = (f"Are these two final answers mathematically equivalent?\nA: {pred}\n"
             f"B: {row['reference']}\nReply with only YES or NO.")
        out = chat("judge", [{"role": "user", "content": q}], max_tokens=300)
        return (10 if "YES" in out["answer"].upper() else 0), out["cost_usd"]
    if t == "multiple_choice":
        return (10 if extract_letter(ans) == str(row["reference"]).strip() else 0), 0.0
    if t == "code_tests":
        return (10 if run_code(ans, row["extra"]) else 0), 0.0
    ctx = ""
    if t == "search_judge":
        ctx = f"SEARCH RESULTS (use these to check facts):\n{search_cache.get(row['prompt'], '')}\n"
    return judge(row["prompt"], ans, ctx)

print("Scoring ready")

Scoring ready


In [9]:
SCORED = f"{OUT}/scored_results.csv"
raw = pd.read_csv(RAW, keep_default_na=False)
rows_by_id = df.set_index("id")

scored_done = set()
if os.path.exists(SCORED):
    sp = pd.read_csv(SCORED, keep_default_na=False)
    scored_done = set(zip(sp["id"], sp["tier"]))

count = 0
for _, r in raw.iterrows():
    if (r["id"], r["tier"]) in scored_done:
        continue
    try:
        s, jcost = score(rows_by_id.loc[r["id"]], str(r["answer"]))
    except Exception as e:
        print("SCORE FAILED", r["id"], r["tier"], str(e)[:150])
        continue
    rec = r.to_dict()
    rec["score"] = s
    rec["judge_cost_usd"] = jcost
    pd.DataFrame([rec]).to_csv(SCORED, mode="a", header=not os.path.exists(SCORED), index=False)
    count += 1
    if count % 20 == 0:
        print(f"{count} scored")

print("Scored this run:", count)

Scored this run: 18


In [10]:
s = pd.read_csv(SCORED, keep_default_na=False)
s = s.merge(df[["id", "category", "difficulty"]], on="id")
s["score"] = s["score"].astype(float)
s["passed"] = s["score"] >= 7

print("=== OVERALL BY TIER ===")
summary = s.groupby("tier").agg(
    prompts=("id", "count"),
    pass_rate=("passed", "mean"),
    total_cost_usd=("cost_usd", "sum"),
    avg_latency_s=("latency_s", "mean"),
).reindex(TIERS).round(4)
print(summary)

print("\n=== PASS RATE BY CATEGORY ===")
print(s.pivot_table(index="category", columns="tier", values="passed", aggfunc="mean")[TIERS].round(2))

print("\n=== RANDOM ROUTING BASELINE ===")
random.seed(0)
pick = {i: random.choice(TIERS) for i in s["id"].unique()}
rnd = s[s.apply(lambda x: pick[x["id"]] == x["tier"], axis=1)]
print("pass_rate:", round(rnd["passed"].mean(), 4), "| total_cost_usd:", round(rnd["cost_usd"].sum(), 4))

order = {"free": 0, "cheap": 1, "premium": 2}
passed = s[s["passed"]].copy()
passed["o"] = passed["tier"].map(order)
best = passed.groupby("id")["o"].min().map({v: k for k, v in order.items()})
labels = df[["id", "category", "difficulty"]].copy()
labels["best_tier"] = labels["id"].map(best).fillna("none")
labels = labels[labels["id"].isin(s["id"].unique())]
print("\n=== CHEAPEST TIER THAT PASSED ===")
print(labels["best_tier"].value_counts())

print("\nJudge cost so far (USD):", round(s["judge_cost_usd"].astype(float).sum(), 4))
labels.to_csv(f"{OUT}/labels.csv", index=False)
summary.to_csv(f"{OUT}/summary.csv")


=== OVERALL BY TIER ===
         prompts  pass_rate  total_cost_usd  avg_latency_s
tier                                                      
free           6     0.6667          0.0000         1.2450
cheap          6     0.8333          0.0051         7.0683
premium        6     0.6667          0.0218         3.5067

=== PASS RATE BY CATEGORY ===
tier       free  cheap  premium
category                       
code        1.0    1.0      1.0
factual     0.0    0.0      0.0
math        1.0    1.0      1.0
reasoning   1.0    1.0      1.0
search      0.0    1.0      0.0
writing     1.0    1.0      1.0

=== RANDOM ROUTING BASELINE ===
pass_rate: 0.8333 | total_cost_usd: 0.0071

=== CHEAPEST TIER THAT PASSED ===
best_tier
free     4
none     1
cheap    1
Name: count, dtype: int64

Judge cost so far (USD): 0.0205
